# Mean-Reversion Structural Forecast to 2045

Monte Carlo mean-reversion forecast projecting each MSA's structural gap back toward its 2015–2018 equilibrium anchor, aggregated nationally on an R&D-weighted basis.

**Requires:** run `01_main_model.ipynb` and `03_regional_analysis.ipynb` first.

In [ ]:
"""
Mean-Reversion Structural Forecast to 2045 — R&D-Weighted National Aggregation
=========================================================================
UPDATED FOR v14b (100-MSA panel -- Hattiesburg, MS and Gulfport-Biloxi,
MS excluded from the ENTIRE main-model pipeline, not just presentation
filtering; see the main model's

(Full original version history retained in git log / docs/methodology.md.)
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

BLUE = "#1a3a5c"
GRAY = "#888888"
RED  = "#d62728"
TEAL = "#17becf"
ORANGE = "#ff7f0e"

LOOCV_CSV = "AvailSFTotal_LOOCV_Residuals.csv"
RESULTS_CSV = "AvailSFTotal_Counterfactual_Results.csv"
BYMSA_ADV_CSV = "AvailSFTotal_AdvWeighted_Actual_vs_Counterfactual_ByMSA.csv"

OUTPUT_MSA_CSV = "MeanReversion_RDWeighted_ByMSA_v14b.csv"
OUTPUT_NATIONAL_CSV = "MeanReversion_RDWeighted_National_2015_2045_v14b.csv"
OUTPUT_DECOMPOSITION_CSV = "MeanReversion_RDWeighted_Equilibrium_Decomposition_v14b.csv"
OUTPUT_TRAJECTORY_PNG = "meanreversion_rdweighted_national_bands_v14b.png"
OUTPUT_LAMBDA_CSV = "MeanReversion_RDWeighted_Lambda_Sensitivity_v14b.csv"

ANCHOR_YEARS = [2015, 2016, 2017, 2018]
LAST_ACTUAL_YEAR = 2023
FORECAST_YEARS = list(range(2023, 2046))

LAMBDA_MEAN = 0.30
LAMBDA_SD = 0.075
LAMBDA_MIN, LAMBDA_MAX = 0.05, 0.60
CONVERGENCE_THRESHOLD = 0.05
CONVERGENCE_Z_FLOOR = 0.5

N_SIMULATIONS = 1000
RNG_SEED = 42

NON_PRESENTATION_MSAS = ['Hattiesburg, MS', 'Gulfport-Biloxi, MS']

rng = np.random.default_rng(RNG_SEED)

# ══════════════════════════════════════════════════════════════════
# 0. SANITY CHECKS — required columns, plus a v14b (100-MSA) check
# ══════════════════════════════════════════════════════════════════
_check = pd.read_csv(RESULTS_CSV)
_required = {'MSA_Name', 'Year', 'Structural_Gap', 'Structural_Gap_RAW', 'Available_SF_Total'}
_missing = _required - set(_check.columns)
if _missing:
    raise SystemExit(
        f"STOPPING -- {RESULTS_CSV} is missing columns this forecast expects: {_missing}. "
        f"This looks like a pre-size-bias-fix export -- re-run the main model first."
    )
_n_msas_results = _check['MSA_Name'].nunique()
if _n_msas_results > 100:
    print(f"WARNING -- {RESULTS_CSV} contains {_n_msas_results} MSAs, not 100. This looks "
          f"like a v14 (102-MSA-trained) export, not v14b. Because this script's core inputs "
          f"(LOOCV_Gap, Structural_Gap) are direct model outputs, running against a v14 export "
          f"will NOT reproduce v14b results -- re-run against true v14b CSVs first.")
del _check

_check_adv = pd.read_csv(BYMSA_ADV_CSV)
_required_adv = {'MSA_Name', 'Year', 'Adv_Weighted_Available_SF_Total'}
_missing_adv = _required_adv - set(_check_adv.columns)
if _missing_adv:
    raise SystemExit(
        f"STOPPING -- {BYMSA_ADV_CSV} is missing columns this script expects: {_missing_adv}. "
        f"Re-run the main model to regenerate the Adv-weighted export first."
    )
_n_msas_adv = _check_adv['MSA_Name'].nunique()
if _n_msas_adv > 100:
    print(f"WARNING -- {BYMSA_ADV_CSV} contains {_n_msas_adv} MSAs, not 100. Same v14-vs-v14b "
          f"concern as above.")
del _check_adv

# ══════════════════════════════════════════════════════════════════
# 1. LOAD
# ══════════════════════════════════════════════════════════════════
loocv = pd.read_csv(LOOCV_CSV)[['MSA_Name', 'Year', 'LOOCV_Gap', 'Available_SF_Total']]
results = pd.read_csv(RESULTS_CSV)[
    ['MSA_Name', 'Year', 'Structural_Gap', 'Structural_Gap_RAW', 'Available_SF_Total']]
rd_weight = pd.read_csv(BYMSA_ADV_CSV)[
    ['MSA_Name', 'Year', 'Adv_Weighted_Available_SF_Total']].rename(
    columns={'Adv_Weighted_Available_SF_Total': 'R&D_Weighted_Available_SF_Total'})

loocv_pres = loocv[~loocv['MSA_Name'].isin(NON_PRESENTATION_MSAS)]

# ══════════════════════════════════════════════════════════════════
# 2. SIZE-BIAS REGRESSION — mechanism unchanged, refit to match the
#    main model exactly. VALUES will differ under v14b because
#    LOOCV_Gap itself comes from a 100-MSA-trained model (see module
#    docstring).
# ══════════════════════════════════════════════════════════════════
size_bias_slope, size_bias_intercept, size_bias_r, size_bias_p, size_bias_stderr = stats.linregress(
    np.log(loocv_pres['Available_SF_Total'].clip(lower=1)), loocv_pres['LOOCV_Gap'])

print(f"{'='*70}")
print(f"SIZE-BIAS REGRESSION (refit to match the main model, {len(loocv_pres)} rows, 100-MSA panel)")
print(f"{'='*70}")
print(f"  LOOCV_Gap = {size_bias_intercept:+.4f} + {size_bias_slope:+.4f} * log(Available_SF_Total)")
print(f"  r={size_bias_r:+.4f}, p={size_bias_p:.4f}")
print(f"  (This should match whatever slope/intercept the main model's own Section 8A")
print(f"   regression reported in its v14b console output; if it doesn't, the LOOCV_Residuals")
print(f"   CSV may not be from the same v14b pipeline run as the Results CSV.)")

def size_bias(log_size):
    return size_bias_intercept + size_bias_slope * log_size

_resid = loocv_pres['LOOCV_Gap'] - size_bias(np.log(loocv_pres['Available_SF_Total'].clip(lower=1)))
LOOCV_SIGMA_RAW = loocv_pres['LOOCV_Gap'].std()
LOOCV_SIGMA = _resid.std()
print(f"\n  LOOCV_SIGMA (raw)      : {LOOCV_SIGMA_RAW:.4f}")
print(f"  LOOCV_SIGMA (corrected): {LOOCV_SIGMA:.4f}  (used throughout this script)")

# ══════════════════════════════════════════════════════════════════
# 3. EQUILIBRIUM (anchor window, raw LOOCV_Gap) + GAP_2023 (corrected,
#    pulled directly -- unchanged mechanism, per-MSA, log-space)
# ══════════════════════════════════════════════════════════════════
anchor_obs = loocv[loocv['Year'].isin(ANCHOR_YEARS)]
anchor_by_msa = anchor_obs.groupby('MSA_Name')['LOOCV_Gap'].apply(list).rename('Anchor_Obs')
equilibrium_point = anchor_obs.groupby('MSA_Name')['LOOCV_Gap'].mean().rename('Equilibrium_Gap_LOOCV_raw')

size_anchor = results[results['Year'].isin(ANCHOR_YEARS)].groupby('MSA_Name')[
    'Available_SF_Total'].mean().rename('Size_Anchor_2015_2018')
size_2023 = results[results['Year'] == LAST_ACTUAL_YEAR].groupby('MSA_Name')[
    'Available_SF_Total'].mean().rename('Size_2023')
gap_2023_corrected = results[results['Year'] == LAST_ACTUAL_YEAR].groupby('MSA_Name')[
    'Structural_Gap'].mean().rename('Gap_2023_corrected')
gap_2023_raw = results[results['Year'] == LAST_ACTUAL_YEAR].groupby('MSA_Name')[
    'Structural_Gap_RAW'].mean().rename('Gap_2023_RAW')
rd_weight_2023 = rd_weight[rd_weight['Year'] == LAST_ACTUAL_YEAR].groupby('MSA_Name')[
    'R&D_Weighted_Available_SF_Total'].mean().rename('RD_Weight_2023')

panel = pd.concat([equilibrium_point, anchor_by_msa, size_anchor, size_2023,
                    gap_2023_corrected, gap_2023_raw, rd_weight_2023], axis=1).dropna(
    subset=['Equilibrium_Gap_LOOCV_raw', 'Size_Anchor_2015_2018', 'Size_2023',
            'Gap_2023_corrected', 'Gap_2023_RAW', 'RD_Weight_2023']
).reset_index()
panel['Presentation_Set'] = ~panel['MSA_Name'].isin(NON_PRESENTATION_MSAS)
panel['log_Size_Anchor'] = np.log(panel['Size_Anchor_2015_2018'].clip(lower=1))
panel['log_Size_2023'] = np.log(panel['Size_2023'].clip(lower=1))

pres = panel[panel['Presentation_Set']].reset_index(drop=True)
print(f"\n{len(pres)} MSAs in presentation set (excludes {NON_PRESENTATION_MSAS})")
if len(pres) > 100:
    print(f"WARNING -- {len(pres)} MSAs in presentation set, expected 100 for v14b.")

# ══════════════════════════════════════════════════════════════════
# 4. RECONCILIATION CHECK — unchanged
# ══════════════════════════════════════════════════════════════════
pres['Bias_2023'] = size_bias(pres['log_Size_2023'])
pres['Reconstructed_Gap_2023'] = pres['Gap_2023_RAW'] - pres['Bias_2023']
_reconciliation_diff = (pres['Reconstructed_Gap_2023'] - pres['Gap_2023_corrected']).abs().max()
print(f"\n{'='*70}")
print("RECONCILIATION CHECK -- Gap_2023 (corrected) vs. RAW minus this script's own fit")
print(f"{'='*70}")
print(f"  Max abs discrepancy: {_reconciliation_diff:.6f} "
      f"({'reconciles (within regression-refit tolerance)' if _reconciliation_diff < 0.01 else 'MISMATCH -- investigate: CSVs may not be from the same run'})")

# ══════════════════════════════════════════════════════════════════
# 5. BRING THE ANCHOR EQUILIBRIUM ONTO THE SAME (CORRECTED) SCALE
#    — unchanged, log-space, per-MSA
# ══════════════════════════════════════════════════════════════════
pres['Bias_Anchor'] = size_bias(pres['log_Size_Anchor'])
pres['Residual_Equilibrium'] = pres['Equilibrium_Gap_LOOCV_raw'] - pres['Bias_Anchor']

# ══════════════════════════════════════════════════════════════════
# 6. DECOMPOSE THE NATIONAL R&D-WEIGHTED EQUILIBRIUM
#    (mechanism unchanged; weights = R&D-relevance, 100-MSA panel)
# ══════════════════════════════════════════════════════════════════
rd_weights_2023 = pres['RD_Weight_2023'].values
rd_weights_2023_norm = rd_weights_2023 / rd_weights_2023.sum()

weighted_eq_raw = np.average(pres['Equilibrium_Gap_LOOCV_raw'], weights=rd_weights_2023_norm)
weighted_size_bias_component = np.average(pres['Bias_Anchor'], weights=rd_weights_2023_norm)
weighted_idiosyncratic_component = np.average(pres['Residual_Equilibrium'], weights=rd_weights_2023_norm)

print(f"\n{'='*70}")
print("DECOMPOSITION OF NATIONAL R&D-WEIGHTED EQUILIBRIUM (anchor window, raw, 100-MSA PANEL)")
print(f"{'='*70}")
print(f"  Total R&D-weighted equilibrium (raw)    : {weighted_eq_raw:+.4f}")
print(f"    - Size-bias component (model habit)   : {weighted_size_bias_component:+.4f} "
      f"({weighted_size_bias_component/weighted_eq_raw:.0%} of total)" if weighted_eq_raw != 0 else "")
print(f"    - Idiosyncratic component (real signal): {weighted_idiosyncratic_component:+.4f}")

pd.DataFrame({
    'Component': ['Total RD-weighted equilibrium (raw)', 'Size-bias component', 'Idiosyncratic component'],
    'Value': [weighted_eq_raw, weighted_size_bias_component, weighted_idiosyncratic_component],
}).to_csv(OUTPUT_DECOMPOSITION_CSV, index=False)
print(f"\nSaved: {OUTPUT_DECOMPOSITION_CSV}")

# ══════════════════════════════════════════════════════════════════
# 7. DEVIATION0 — unchanged mechanism, per-MSA, log-space
# ══════════════════════════════════════════════════════════════════
pres['Equilibrium_ForForecast'] = pres['Residual_Equilibrium'] + pres['Bias_2023']
pres['Deviation0'] = pres['Gap_2023_corrected'] - pres['Equilibrium_ForForecast']
pres['Already_Converged_2023'] = pres['Deviation0'].abs() <= (CONVERGENCE_Z_FLOOR * LOOCV_SIGMA)

print(f"\nDeviation0 (R&D-weighted national aggregation), presentation set "
      f"(RD-weighted mean): {np.average(pres['Deviation0'], weights=rd_weights_2023_norm):+.4f}")
print(f"Already-converged: {pres['Already_Converged_2023'].sum()} of {len(pres)} MSAs")

pres[['MSA_Name', 'Equilibrium_Gap_LOOCV_raw', 'Bias_Anchor', 'Residual_Equilibrium',
      'Gap_2023_corrected', 'Gap_2023_RAW', 'Bias_2023', 'Deviation0',
      'Already_Converged_2023', 'RD_Weight_2023']].sort_values(
    'Deviation0', key=lambda s: s.abs(), ascending=False
).to_csv(OUTPUT_MSA_CSV, index=False)
print(f"Saved: {OUTPUT_MSA_CSV}")

# ══════════════════════════════════════════════════════════════════
# 8. MONTE CARLO — per-MSA mechanism unchanged; national aggregation
#    weights = R&D-relevance, on the 100-MSA panel
# ══════════════════════════════════════════════════════════════════
n_years = len(FORECAST_YEARS)
n_pres = len(pres)
weighted_forecast_sims = np.zeros((N_SIMULATIONS, n_years))
n_star_sims = np.zeros(N_SIMULATIONS)

print(f"\nRunning {N_SIMULATIONS} Monte Carlo simulations (R&D-weighted national aggregation, 100-MSA panel)...")
for s in range(N_SIMULATIONS):
    lam_s = np.clip(rng.normal(LAMBDA_MEAN, LAMBDA_SD), LAMBDA_MIN, LAMBDA_MAX)
    n_star_sims[s] = np.ceil(np.log(CONVERGENCE_THRESHOLD) / np.log(1 - lam_s))

    eq_boot = np.empty(n_pres)
    for i, obs in enumerate(pres['Anchor_Obs']):
        obs = np.array(obs)
        resampled = rng.choice(obs, size=len(obs), replace=True)
        eq_boot[i] = resampled.mean()
    residual_eq_s = eq_boot - pres['Bias_Anchor'].values

    gap2023_s = pres['Gap_2023_corrected'].values + rng.normal(0, LOOCV_SIGMA, size=n_pres)

    eq_forecast_s = residual_eq_s + pres['Bias_2023'].values
    dev0_s = gap2023_s - eq_forecast_s

    for t_idx, year in enumerate(FORECAST_YEARS):
        n = year - LAST_ACTUAL_YEAR
        gap_forecast_s = eq_forecast_s + dev0_s * (1 - lam_s) ** n
        weighted_forecast_sims[s, t_idx] = np.average(gap_forecast_s, weights=rd_weights_2023_norm)

print("Monte Carlo complete.")

national_forecast_bands = pd.DataFrame({
    'Year': FORECAST_YEARS,
    'RDWeighted_Mean_Gap_p50': np.percentile(weighted_forecast_sims, 50, axis=0),
    'RDWeighted_Mean_Gap_p05': np.percentile(weighted_forecast_sims, 5, axis=0),
    'RDWeighted_Mean_Gap_p95': np.percentile(weighted_forecast_sims, 95, axis=0),
})

n_star_p50 = np.percentile(n_star_sims, 50)
n_star_p05 = np.percentile(n_star_sims, 5)
n_star_p95 = np.percentile(n_star_sims, 95)
year_p50 = LAST_ACTUAL_YEAR + int(n_star_p50)
year_p05 = LAST_ACTUAL_YEAR + int(n_star_p05)
year_p95 = LAST_ACTUAL_YEAR + int(n_star_p95)

print(f"\n{'='*70}")
print("TIME TO CLOSE 95% OF THE R&D-WEIGHTED NATIONAL DEVIATION (100-MSA PANEL)")
print(f"{'='*70}")
print(f"  Median       : {n_star_p50:.0f} years -> {year_p50}")
print(f"  90% interval : [{n_star_p05:.0f}, {n_star_p95:.0f}] years -> [{year_p05}, {year_p95}]")
print(f"  NOTE: this timeline is driven ENTIRELY by LAMBDA_MEAN/LAMBDA_SD "
      f"({LAMBDA_MEAN}, {LAMBDA_SD}), an assumed reversion-speed prior -- identical mechanism "
      f"to the unweighted version, since lambda is applied per-MSA before any aggregation, and "
      f"the R&D-weighting change only affects HOW the (now 100) per-MSA forecasts are combined "
      f"into one number, not the per-MSA reversion speed itself. The NUMBER ITSELF, however, "
      f"can still differ from a v14 run, since Deviation0 and the equilibrium level both "
      f"depend on LOOCV_Gap/Structural_Gap from the retrained model. See the lambda "
      f"sensitivity table below.")

# ══════════════════════════════════════════════════════════════════
# 8B. LAMBDA SENSITIVITY — deterministic, purely a function of lambda
#     and CONVERGENCE_THRESHOLD, independent of weighting scheme AND
#     independent of the v14-vs-v14b MSA count
# ══════════════════════════════════════════════════════════════════
LAMBDA_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]

lambda_sensitivity = pd.DataFrame({'Lambda': LAMBDA_GRID})
lambda_sensitivity['Years_to_95pct_Closure'] = lambda_sensitivity['Lambda'].apply(
    lambda lam: int(np.ceil(np.log(CONVERGENCE_THRESHOLD) / np.log(1 - lam))))
lambda_sensitivity['Year'] = LAST_ACTUAL_YEAR + lambda_sensitivity['Years_to_95pct_Closure']
lambda_sensitivity['Is_Assumed_Mean'] = np.isclose(lambda_sensitivity['Lambda'], LAMBDA_MEAN)
lambda_sensitivity.to_csv(OUTPUT_LAMBDA_CSV, index=False)

print(f"\n{'='*70}")
print("LAMBDA SENSITIVITY — years to close 95% of the deviation, by assumed reversion speed")
print(f"{'='*70}")
print(lambda_sensitivity.to_string(index=False))
print(f"\nSaved: {OUTPUT_LAMBDA_CSV}")

# ══════════════════════════════════════════════════════════════════
# 9. NATIONAL AGGREGATE — actual (weighted by R&D-relevance, 100-MSA
#    panel, 2015-2023)
# ══════════════════════════════════════════════════════════════════
actual_panel = pd.read_csv(RESULTS_CSV)
actual_panel = actual_panel[
    actual_panel['Year'].between(2015, LAST_ACTUAL_YEAR) &
    ~actual_panel['MSA_Name'].isin(NON_PRESENTATION_MSAS)
].copy()
actual_panel = actual_panel.merge(
    rd_weight, on=['MSA_Name', 'Year'], how='left')
national_actual = actual_panel.dropna(subset=['R&D_Weighted_Available_SF_Total']).groupby('Year').apply(
    lambda g: pd.Series({'RDWeighted_Mean_Gap': np.average(
        g['Structural_Gap'], weights=g['R&D_Weighted_Available_SF_Total'])})
).reset_index()

national_export = national_actual.merge(national_forecast_bands, on='Year', how='outer')
national_export.to_csv(OUTPUT_NATIONAL_CSV, index=False)
print(f"\nSaved: {OUTPUT_NATIONAL_CSV}")

national_eq_forecast_weighted = np.average(pres['Equilibrium_ForForecast'], weights=rd_weights_2023_norm)
print(f"National R&D-weighted long-run target (at 2023 R&D-relevance weights, 100-MSA panel): "
      f"{national_eq_forecast_weighted:+.4f}")

# ══════════════════════════════════════════════════════════════════
# 10. FIGURE — title completed (was truncated), annotation restored
#     (was commented-out dead code) and updated for v14b
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(national_actual['Year'], national_actual['RDWeighted_Mean_Gap'],
        'o-', color=BLUE, lw=2.2, ms=5, label='Actual (R&D-weighted Structural Gap)', zorder=5)
ax.fill_between(national_forecast_bands['Year'],
                 national_forecast_bands['RDWeighted_Mean_Gap_p05'],
                 national_forecast_bands['RDWeighted_Mean_Gap_p95'],
                 color=RED, alpha=0.15, label='90% band (Monte Carlo)')
ax.plot(national_forecast_bands['Year'], national_forecast_bands['RDWeighted_Mean_Gap_p50'],
        '-', color=RED, lw=2, label='Forecast median (R&D-weighted)')
ax.axhline(national_eq_forecast_weighted, color=TEAL, lw=1.2, ls=':',
           label=f'R&D-weighted long-run target ({national_eq_forecast_weighted:+.4f})')
ax.axvspan(2019.5, LAST_ACTUAL_YEAR + 0.5, alpha=0.06, color='red')
ax.axvline(year_p50, color=TEAL, lw=1.3, ls='-.', alpha=0.8,
           label=f'Median 95%-closure: {year_p50} (90% CI: {year_p05}-{year_p95})')

ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Mean Structural Gap (log units)", fontsize=11)
ax.set_title(
    "National Structural Gap — R&D-Weighted Mean-Reversion Forecast to 2045\n"
    "[SIZE-CORRECTED, TRUE-NATIONAL-LQ, 100-MSA PANEL]",
    fontsize=11.5, fontweight='bold')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.25, ls=':')
ax.set_xlim(2014.5, 2046)
ax.annotate(
    f"Reversion speed assumed, not estimated: \u03bb~N({LAMBDA_MEAN}, {LAMBDA_SD}).\n"
    f"See {OUTPUT_LAMBDA_CSV} for how the closure year shifts across \u03bb={LAMBDA_GRID[0]}-{LAMBDA_GRID[-1]}.\n"
    f"Weighting is R&D-relevance (R&D_Weighted_Available_SF_Total, true-national-LQ), "
    f"100-MSA panel.",
    xy=(0.02, 0.02), xycoords='axes fraction', fontsize=7.5, color='#666666',
    ha='left', va='bottom', style='italic')
plt.tight_layout()
plt.savefig(OUTPUT_TRAJECTORY_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"\nSaved: {OUTPUT_TRAJECTORY_PNG}")

print(f"\n{'='*60}")
print("DONE — Mean-Reversion Forecast, R&D-Weighted National Aggregation (v14b, 100-MSA panel)")
print(f"{'='*60}")
print(f"  Headline timeline (assumed \u03bb~N({LAMBDA_MEAN},{LAMBDA_SD})): median {year_p50}, "
      f"90% CI [{year_p05}, {year_p95}]")
print(f"  Lambda sensitivity (assumption-free range check): see {OUTPUT_LAMBDA_CSV}")